In [0]:
# =============================================================================
# EA Real-Time Daily Archive Ingest
# =============================================================================
# Notebook:   10_ea_rt_daily_archive.py
# Schema:     prd_dash_lab.flood_forecasting_unrestricted
# Table:      ea_rt_readings_bronze
# Source:     EA Real-Time Flood Monitoring Archive
#             https://environment.data.gov.uk/flood-monitoring/archive
# Schedule:   Daily, after 22:00. Triggered via Databricks scheduled job.
#
# Purpose:
#   Downloads yesterday's readings-full CSV from the EA archive and appends
#   it to the Bronze Delta table. After landing the readings, calls
#   09_ea_rt_measure_register to register any new measures introduced by
#   yesterday's file.
#
# Archive availability:
#   Each day's archive file is generated at 22:00 the following day to allow
#   late telemetry to arrive. This notebook therefore targets (today - 1).
#   Schedule the job to run at 23:00 or later to ensure the file exists.
#
# Idempotency:
#   If yesterday's partition already exists in Bronze (e.g. from a re-run),
#   the write uses replaceWhere to overwrite it cleanly rather than duplicate.
#
# Attribution (OGL):
#   "this uses Environment Agency flood and river level data from the
#    real-time data API (Beta)"
# =============================================================================

import requests
import io
from datetime import datetime, timedelta, timezone, date

import pandas as pd

from pyspark.sql import functions as F

# =============================================================================
# CONFIGURATION
# =============================================================================

CATALOG        = "prd_dash_lab"
SCHEMA         = "flood_forecasting_unrestricted"
BRONZE_TABLE   = "ea_rt_readings_bronze"
REGISTER_NB    = "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks/jobs/09_ea_rt_measure_register"

FULL_BRONZE_NAME = f"{CATALOG}.{SCHEMA}.{BRONZE_TABLE}"

ARCHIVE_BASE           = "https://environment.data.gov.uk/flood-monitoring/archive"
DOWNLOAD_TIMEOUT_SECONDS = 300

# CSV column name -> Bronze column name
CSV_COLUMN_MAP = {
    "dateTime":         "date_time",
    "date":             "reading_date",
    "measure":          "measure_uri",
    "station":          "station_uri",
    "label":            "label",
    "stationReference": "station_reference",
    "parameter":        "parameter",
    "qualifier":        "qualifier",
    "datumType":        "datum_type",
    "period":           "period",
    "unitName":         "unit_name",
    "valueType":        "value_type",
    "value":            "value",
}


In [0]:
# =============================================================================
# STEP 1: DETERMINE TARGET DATE
# =============================================================================
# Target is yesterday -- the most recent complete archive file.
# The file for a given day is not generated until 22:00 the following day,
# so today's file does not yet exist when this job runs.

target_date = date.today() - timedelta(days=1)
date_str    = target_date.strftime("%Y-%m-%d")
url         = f"{ARCHIVE_BASE}/readings-full-{date_str}.csv"

print(f"Target date: {date_str}")
print(f"URL: {url}")


In [0]:
# =============================================================================
# STEP 2: CHECK WHETHER THIS DATE IS ALREADY IN BRONZE
# =============================================================================
# Protects against double-processing if the job is triggered twice.
# replaceWhere in Step 4 also handles this, but an early check avoids
# a redundant 130MB download.

try:
    existing_dates = {
        row["reading_date"]
        for row in spark.sql(f"""
            SELECT DISTINCT reading_date FROM {FULL_BRONZE_NAME}
        """).collect()
    }
    already_loaded = target_date in existing_dates
except Exception:
    # Bronze table does not exist yet -- should not happen after the backfill
    # but handle gracefully
    already_loaded = False

if already_loaded:
    print(f"Partition {date_str} already present in Bronze. Re-downloading to ensure completeness.")
    dbutils.notebook.exit("success")
else:
    print(f"Partition {date_str} not yet in Bronze. Proceeding with download.")


In [0]:
# =============================================================================
# STEP 3: DOWNLOAD THE ARCHIVE FILE
# =============================================================================

try:
    response = requests.get(url, timeout=DOWNLOAD_TIMEOUT_SECONDS)
    response.raise_for_status()
except requests.exceptions.HTTPError:
    if response.status_code == 404:
        # File not yet generated -- exit cleanly rather than failing the job
        print(f"WARNING: Archive file for {date_str} not found (HTTP 404).")
        print("The file may not yet have been generated. Re-run after 22:00.")
        dbutils.notebook.exit("success")
    else:
        raise

file_size_mb = len(response.content) / (1024 * 1024)
print(f"Downloaded {file_size_mb:.1f} MB")

In [0]:
# =============================================================================
# STEP 4: PARSE AND LAND TO BRONZE
# =============================================================================

now_utc = datetime.now(timezone.utc)

# Parse CSV -- read all columns as string; type casting happens in Spark
pdf = pd.read_csv(
    io.BytesIO(response.content),
    dtype=str,
    low_memory=False
)

print(f"Rows: {len(pdf):,}")

# Rename to Bronze convention; drop any unexpected extra columns
pdf = pdf.rename(columns=CSV_COLUMN_MAP)
pdf = pdf[[c for c in CSV_COLUMN_MAP.values() if c in pdf.columns]]
pdf["ingested_at"] = now_utc

# Convert to Spark and cast types
sdf = (
    spark.createDataFrame(pdf)
    .withColumn("date_time",    F.to_timestamp("date_time"))
    .withColumn("reading_date", F.to_date("reading_date"))
    .withColumn("period",       F.expr("try_cast(period AS INT)"))
    .withColumn("value",        F.expr("try_cast(value AS DOUBLE)"))
    .withColumn("ingested_at",  F.col("ingested_at").cast("timestamp"))
)

# Write to Bronze -- replaceWhere overwrites this date's partition only
(
    sdf.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"reading_date = '{date_str}'")
    .partitionBy("reading_date")
    .saveAsTable(FULL_BRONZE_NAME)
)

print(f"Written to Bronze (partition: {date_str})")


In [0]:
# =============================================================================
# STEP 5: REGISTER ANY NEW MEASURES
# =============================================================================
# Check yesterday's Bronze partition for measure URIs not yet in the register.
# Scanning one partition is fast. If no new measures exist, skip the register
# notebook call entirely -- avoids a full Bronze table scan and any API calls.

new_measures = spark.sql(f"""
    SELECT DISTINCT b.measure_uri
    FROM {FULL_BRONZE_NAME} b
    LEFT JOIN {CATALOG}.{SCHEMA}.ea_rt_measure_register r
        ON b.measure_uri = r.measure_uri
    WHERE b.reading_date = '{date_str}'
    AND r.measure_uri IS NULL
""")

new_measure_count = new_measures.count()
print(f"New measures in yesterday's file: {new_measure_count}")

if new_measure_count > 0:
    print("New measures found. Calling measure register notebook...")
    register_result = dbutils.notebook.run(
        path=REGISTER_NB,
        timeout_seconds=3600,  # station API crawl for new measures can take time
        arguments={}
    )
    if register_result != "success":
        raise Exception(f"Measure register notebook failed with: {register_result}")
    print("Measure register updated.")
else:
    print("No new measures. Register notebook skipped.")


In [0]:
# =============================================================================
# STEP 6: SUMMARY
# =============================================================================

spark.sql(f"""
    SELECT
        COUNT(*)                      AS row_count,
        COUNT(DISTINCT measure_uri)   AS measure_count,
        COUNT(DISTINCT station_reference) AS station_count,
        SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS null_values
    FROM {FULL_BRONZE_NAME}
    WHERE reading_date = '{date_str}'
""").show(truncate=False)


In [0]:
# =============================================================================
# SIGNAL COMPLETION
# =============================================================================

print(f"Daily archive ingest complete for {date_str}.")
dbutils.notebook.exit("success")
